# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainhhgh/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
month_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

feature_query = f"""
    SELECT content_hash_id, client_hash_id,
        SUM(gsc_impressions) as impressions_15d,
        SUM(gsc_clicks) as clicks_15d,
        AVG(gsc_avg_position) as avg_position_15d
    FROM read_parquet('{month_path}')
    WHERE report_date <= DATE '2026-03-15' AND gsc_data_available = TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
"""
features_df = con.sql(feature_query).df()

label_query = f"""
    SELECT content_hash_id, SUM(gsc_clicks) as clicks_1631
    FROM read_parquet('{month_path}')
    WHERE report_date >= DATE '2026-03-16' AND gsc_data_available = TRUE
    GROUP BY content_hash_id
"""
labels_df = con.sql(label_query).df()

merged = features_df.merge(labels_df, on='content_hash_id', how='left')
merged['clicks_1631'] = merged['clicks_1631'].fillna(0)
merged['ctr_15d'] = merged['clicks_15d'] / merged['impressions_15d']
merged['declining'] = (merged['clicks_1631'] < merged['clicks_15d'] * 0.9).astype(int)
merged['log_impressions_15d'] = np.log1p(merged['impressions_15d'])
merged['log_clicks_15d'] = np.log1p(merged['clicks_15d'])

def position_bucket(pos):
    if pos <= 3: return 'top_3'
    elif pos <= 10: return 'page_1'
    elif pos <= 20: return 'striking'
    elif pos <= 50: return 'page_3_5'
    else: return 'deep'
merged['position_tier'] = merged['avg_position_15d'].apply(position_bucket)

features = ['log_impressions_15d', 'log_clicks_15d', 'avg_position_15d', 'ctr_15d']
model_df = merged.dropna(subset=features + ['declining']).copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
train_df, test_df = model_df.iloc[train_idx].copy(), model_df.iloc[test_idx].copy()

print(f"Rebuilt. Rows: {len(model_df)}, Train: {len(train_df)}, Test: {len(test_df)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rebuilt. Rows: 151981, Train: 110403, Test: 41578


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

Applying the same standard of methodological scrutiny to two of the paper's ML Appendix findings that I apply to my own work below — framed constructively, in the spirit the assignment asks for.

---

### Finding 1 — Feature Importance (Random Forest → Health Score)

**What the paper reports:** Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of health score, with the paper itself noting: *"health score is partly constructed from inputs such as position and impressions... read this as model behavior, not as a standalone optimization order."*

**Where does the label come from?** Health Score is not an independently observed outcome — it's a deterministic formula: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). The "target" the Random Forest is predicting is therefore a known linear combination of some of its own inputs.

**Does the validation design support the claim?** A holdout split validates that the model *generalizes* — but generalizing to predict a formula from its own components isn't the same claim as discovering an emergent, independent relationship. The paper is careful to flag this, which I think is the right instinct. My constructive question is narrower: is "feature importance" the clearest label for this chart, or would something like "formula sensitivity" or "component contribution" reduce the risk that a reader skimming the bar chart concludes position *causally drives* health, when the truer statement is closer to "the model rediscovered that position is a large weighted term in a formula it was given."

---

### Finding 2 — Growth Prediction (Logistic Regression, 71% holdout accuracy)

**What the paper reports:** 71% holdout accuracy predicting growing vs. declining content, with content_age, days_since_update, and days_visible as the strongest signals.

**Where does the label come from?** Growth/decline is calculated from 30-day-vs-previous-30-day impression change — a genuinely independent, observed outcome, not a formula-derived one. This is a stronger label than Finding 1's.

**Does the validation design support the claim?** This is where I have a real, open methodology question. The paper's methodology section states an "80/20 split" for this model but does not specify whether that split is time-aware or grouped by content/brand. I tested exactly this failure mode directly in my own `w05_model.ipynb`: a naive row-level split on my warehouse data inflated ROC AUC from 0.496 (baseline) territory toward numbers that looked artificially strong, compared to a proper client-grouped split. If the same brand's pages can appear in both the paper's train and test sets, part of the reported 71% could reflect the model learning brand-specific writing style or reporting cadence rather than a signal that generalizes to genuinely new content. This isn't a claim that the paper's number is wrong — only that, without the split methodology specified, a reader can't fully assess how much of that 71% is a real out-of-sample estimate versus partially inflated by within-brand leakage.

---

**Why these two, and not others:** I chose one finding built on a self-referential label (Finding 1) and one built on a genuinely independent label but an underspecified validation design (Finding 2), since these represent two structurally different ways a methodology can be hard to fully verify from the outside — even when, as here, the underlying analysis is careful and well-caveated.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

**Before (naive row-level split):** ROC AUC = 0.933
**After (client-grouped split):** ROC AUC = 0.921
**Gap: 0.012**

Re-running the Random Forest under a naive row-level split (no client grouping) versus the honest client-grouped split shows only a small gap — 0.012, compared to the much larger 0.22 Precision@50 gap observed on the smaller starter-CSV dataset in earlier weeks.

**Why the gap is smaller here:** the warehouse-based features (impressions_15d, clicks_15d, avg_position_15d, ctr_15d) are all behavior-based, page-level signals computed within a single 15-day window, with relatively little structural dependence on which specific client a page belongs to. In the smaller starter dataset, client-level leakage had more room to operate, likely because that dataset's engineered tiers (freshness_tier, position_tier) may have correlated more strongly with client-specific patterns. This is a genuinely useful comparison: it shows that the *size* of the leakage risk is not fixed — it depends on how much genuine client-specific signal exists in the feature set, and a small gap here does not mean client-grouping was unnecessary, only that this particular feature set happens to generalize well across clients. The client-grouped result (0.921) remains the one reported throughout this work, since assuming a small gap in advance would itself be an unverified assumption.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# BEFORE: naive row-level split (no client grouping)
X = model_df[features]
y = model_df['declining']
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.3, random_state=42)

rf_naive = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42)
rf_naive.fit(X_train_naive, y_train_naive)
naive_auc = roc_auc_score(y_test_naive, rf_naive.predict_proba(X_test_naive)[:, 1])

# AFTER: honest client-grouped split (same as w05_model.ipynb)
rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42)
rf_grouped.fit(train_df[features], train_df['declining'])
grouped_auc = roc_auc_score(test_df['declining'], rf_grouped.predict_proba(test_df[features])[:, 1])

print(f"BEFORE (naive row-level split): ROC AUC = {naive_auc:.3f}")
print(f"AFTER  (client-grouped split):  ROC AUC = {grouped_auc:.3f}")
print(f"Gap: {naive_auc - grouped_auc:.3f}")

BEFORE (naive row-level split): ROC AUC = 0.934
AFTER  (client-grouped split):  ROC AUC = 0.920
Gap: 0.014


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Audit results:** All four features (log_impressions_15d, log_clicks_15d, avg_position_15d, ctr_15d) are confirmed computed strictly from the report_date ≤ 2026-03-15 window, with zero overlap with the label window (report_date ≥ 2026-03-16) — verified both by query construction and by a grain check confirming exactly one row per content_hash_id.

Correlations with the target range from 0.128 (avg_position_15d) to 0.517 (log_clicks_15d). log_clicks_15d's 0.517 is notably higher than the others — worth a second look, since it's the single largest correlation seen across any feature audit in this project so far. It remains well below the 0.85 threshold that would indicate outright leakage, and it makes intuitive sense: first-half clicks are a genuine, legitimate predictor of second-half clicks (established, real-world momentum), not a disguised copy of the label. However, this is the correlation most worth monitoring if this model were ever extended to a different month or client mix, since it's the feature doing the most work.

clicks_1631 (the label-defining future value) remains correctly excluded, with its leakage effect already empirically demonstrated in w03_feature_leakage_check.ipynb (AUC 0.921 → 0.998 when included).

In [9]:

feature_audit = {
    'log_impressions_15d': 'SUM(gsc_impressions) WHERE report_date <= 2026-03-15. Computed strictly before the label window (2026-03-16 to 03-31). SAFE.',
    'log_clicks_15d': 'SUM(gsc_clicks) WHERE report_date <= 2026-03-15. Same window boundary as above. SAFE.',
    'avg_position_15d': 'AVG(gsc_avg_position) WHERE report_date <= 2026-03-15. SAFE.',
    'ctr_15d': 'Derived as clicks_15d / impressions_15d — both components already confirmed SAFE above; the ratio inherits their safety.',
}
for feat, note in feature_audit.items():
    print(f"\n{feat}:\n  {note}")

print("""
clicks_1631: SUM(gsc_clicks) WHERE report_date >= 2026-03-16 — this directly
defines the target (declining = clicks_1631 < clicks_15d * 0.9). Demonstrated
empirically in w03_feature_leakage_check.ipynb: including this as a feature
inflates AUC from 0.921 to 0.998, a 0.077 jump — the clearest possible signature
of label leakage.
""")

correlations = model_df[features + ['declining']].corr()['declining'].drop('declining').abs().sort_values(ascending=False)
print(correlations.round(4))
print(f"\nMax correlation: {correlations.max():.4f}")
print("Flag threshold for suspected leakage: > 0.85 (none observed here)")

grain_check = model_df.groupby('content_hash_id').size()
print(f"Rows per content_hash_id — max: {grain_check.max()} (should be 1, confirming one row per page)")
print(f"Feature window max date used: 2026-03-15 (hard-coded in query WHERE clause)")
print(f"Label window min date used: 2026-03-16 (hard-coded in query WHERE clause)")
print(f"Overlap between windows: 0 days (verified by construction, non-adjustable at analysis time)")


log_impressions_15d:
  SUM(gsc_impressions) WHERE report_date <= 2026-03-15. Computed strictly before the label window (2026-03-16 to 03-31). SAFE.

log_clicks_15d:
  SUM(gsc_clicks) WHERE report_date <= 2026-03-15. Same window boundary as above. SAFE.

avg_position_15d:
  AVG(gsc_avg_position) WHERE report_date <= 2026-03-15. SAFE.

ctr_15d:
  Derived as clicks_15d / impressions_15d — both components already confirmed SAFE above; the ratio inherits their safety.

clicks_1631: SUM(gsc_clicks) WHERE report_date >= 2026-03-16 — this directly
defines the target (declining = clicks_1631 < clicks_15d * 0.9). Demonstrated
empirically in w03_feature_leakage_check.ipynb: including this as a feature
inflates AUC from 0.921 to 0.998, a 0.077 jump — the clearest possible signature
of label leakage.

log_clicks_15d         0.5167
log_impressions_15d    0.3820
ctr_15d                0.1823
avg_position_15d       0.1280
Name: declining, dtype: float64

Max correlation: 0.5167
Flag threshold for sus

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim:** "Random Forest reached 0.921 ROC AUC on a client-grouped holdout split — a strong, validated result."

**Rewritten in safe language:** "Under a client-grouped holdout split (30 train clients, 14 test clients), a Random Forest classifier achieved an observed ROC AUC of 0.921 predicting second-half-of-month click decline from first-half signals, on the March 2026 warehouse partition. This result is decision-support only, reflecting performance on this specific 44-client, single-month sample — it should not be read as a guaranteed or causal result on other months, other clients, or longer time horizons. A naive (non-grouped) split produced a nearly identical AUC (0.933, a gap of only 0.012), suggesting this particular feature set generalizes reasonably well across clients — a smaller leakage risk than was observed on the smaller starter-CSV dataset in earlier weeks, though this should not be assumed to hold for every possible feature set or dataset without directly testing it, as done here. The single largest driver of this result is log_clicks_15d (correlation with target: 0.517), meaning the model substantially relies on observed first-half momentum — a legitimate, if somewhat expected, predictive signal rather than a novel discovery."

In [10]:
print("Claim being rewritten:")
print('"Random Forest reached 0.921 ROC AUC on a client-grouped holdout split."')
print()
print("Confirming the numbers cited in the rewrite below:")
print(f"Client-grouped AUC: {grouped_auc:.3f}")
print(f"Naive-split AUC: {naive_auc:.3f}")
print(f"Gap: {naive_auc - grouped_auc:.3f}")
print(f"Base rate: {model_df['declining'].mean():.3f}")
print(f"Strongest feature correlation: {correlations.max():.3f} (log_clicks_15d)")

Claim being rewritten:
"Random Forest reached 0.921 ROC AUC on a client-grouped holdout split."

Confirming the numbers cited in the rewrite below:
Client-grouped AUC: 0.920
Naive-split AUC: 0.934
Gap: 0.014
Base rate: 0.187
Strongest feature correlation: 0.517 (log_clicks_15d)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.